# Sprint 3 - Manipulación de Datos I (Sesiones)

En este caso de estudio se busca prácticar los fundamentos de manipulación de datos con **Pandas**. Adicionalmente, se va a introducir los primeros criterios para un adecuado ordenamiento de proyectos de análisis de datos:

* Entendimiento del contexto
* Entendimiento de los datos
* Preparación de los datos
* Análisis de datos

## Entendimiento del contexto

El ranking de selecciones masculinas FIFA es un mecanismo que utiliza la entidad rectora del fútbol mundial para clasificar a los distintos equipos asociados. El mismo inició su medición en el año 1993, previo al mundial de Estados Unidos y se lo ha venido actualizando periódicamente en base a los resultados obtenidos en cada uno de los enfrentamientos oficiales y amistosos en las fechas designadas como FIFA (generalmente, esta actualización se la hace de manera mensual).

Actualmente este ranking sirve no solamente para informar a los aficionados respecto a cuáles son las mejores selecciones a nivel mundial, sino que es el principal criterio utilizado para definir los grupos en las citas mundialistas que se realizan cada 4 años.

Es importante que sepas además que este ranking ha tenido cambios importantes a nivel de su metodología de cálculo en Enero de 1999, Agosto 2006, Agosto 2011 y Agosto 2018.

Para un mayor detalle asociado a qué es y cómo se calcula este ranking, te sugiero revisar el siguiente link: https://en.wikipedia.org/wiki/FIFA_Men%27s_World_Ranking.

## Entendimiento de los datos

Lo primero que te sugiero hacer al iniciar cualquier proyecto de análisis de datos es importar las librerías y módulos con los que vas a trabajar. En este caso en concreto vamos cargar: 

* **Pandas** cuyo objetivo es proporcionar funciones y métodos para un eficiente manejo de tablas de datos (también conocidos como *dataframes* o *datasets*).
* **Numpy** cuyo objetivo es proporcionar funciones y métodos para análisis numérico.

In [1]:
import pandas as pd #pd es el pseudónimo con el que llamaremos a pandas en todo el documento
import numpy as np #np es el pseudónimo con el que llmaremos a numpy en todo el documento

Vale señalar que si el código anterior genera un error que indica que el módulo *pandas* no existe, deberás instalar por única vez esta librería ejecutando el siguiente comando en una celda previa:

```py
!pip install pandas
```

A continuación, conviene cargar los datos del archivo **fifa_rank.csv** con los que se vas a trabajar utilizando la función `read_csv`.

In [ ]:
df_fifa = pd.read_csv("fifa_rank.csv", sep = ",") 
#El argumento sep no es obligatorio pero sirve para especificar el caracter con que los datos están separados en el archivo

El dataset **fifa_rank** contiene cerca de 58 mil filas y 6 columnas detalladas a continuación:

* Country: Nombre del pais afiliado a la FIFA cuya selección de fútbol es puntuada.
* Confederation: Confederación a la que pertenece de acuerdo a la división de la FIFA.
* Rank_Date: Fecha de publicación del ranking FIFA entre marzo de 1993 y junio de 2018.
* Points_Old_Version: Puntos obtenidos conforme metodologías de cálculo anteriores al 2011 
* Ponts_New_Version: Puntos obtenidos conforme la metodología posterior a 2011. 
* rank: Posición en el ranking oficial de la FIFA dados los puntos calculados.

Una buena práctica al momento de cargar los datos es observar sus principales características a fin de establecer un plan de tratamiento acorde al dataset y su contexto. Esto resulta de vital importancia puesto que cada conjunto de datos es único y tiene sus propias particularidades. En concreto, interesa ver al menos lo siguiente:

* Estructura de los datos (número de filas y columnas)
* Nombres de las columnas
* Existencia de valores perdidos en cada columna
* Tipos de variables en cada columna
* Aspectos particulares de cada columna

Con este propósito, utiliza el método `info` para un primer vistazo al dataset.

In [3]:
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             57797 non-null  object 
 1   Confederation       54815 non-null  object 
 2   Rank_Date           57797 non-null  object 
 3   Points_Old_Version  40382 non-null  float64
 4   Points_New_Version  57797 non-null  object 
 5   rank                57797 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 2.6+ MB


A continuación algunos aspectos detectados:

* Las dimensiones de la tabla son correctas respecto a lo indicado en la descripción heca previamente.
* Los nombres de las columnas tienen mayusculas y esto puede ser problemático en las siguientes etapas del proyecto.
* Existen valores perdidos en las columnas confederation y Points_Old_Version.
* El tipo de variable de Points_New_Version es **object** (equivalente a *string*) lo cual no hace sentido considerando que deberías ver números.
* El tipo de variable de Rank_Date igualmente es **object**, lo cual tampoco hace sentido puesto que aquí deberías ver fechas.
* Al respecto de esto, que una fecha se encuentre en su tipología adecuada es coherente ya que usualmente va a interesar extraer sus componentes (i.e año, mes, dias, etc.)

Profundiza en tu conocimiento del dataset utilizando ahora el método `sample` con el argumento 15, a fin de extraer una muestra aleatoria de los datos de tamaño 15.

In [4]:
df_fifa.sample(15)

,Country,Confederation,Rank_Date,Points_Old_Version,Points_New_Version,rank
49411,Mexico,CONCACAF,2015-03-12,NaN,935.23,21
23733,Cuba,CONCACAF,2004-09-01,525.0,ND,74
35840,Greece,UEFA,2009-09-02,1001.0,ND,12
2061,Tunisia,CAF,1994-10-25,42.0,ND,30
8950,Algeria,CAF,1998-07-15,43.0,ND,63
50550,St Vincent and the Grenadines,CONCACAF,2015-08-06,NaN,268.36,115
37129,Poland,UEFA,2010-03-31,540.0,ND,59
39621,Venezuela,CONMEBOL,2011-04-13,505.0,ND,67
8861,Nepal,AFC,1998-05-20,6.0,ND,166
47950,Scotland,UEFA,2014-08-14,NaN,738.33,28


Considera los siguientes aspectos observados:

* En el caso de los valores perdidos de Points_Old_Version, destaca que los mismos parecen estár asociados a fechas en las que ya no se utilizaban metodologías antiguas, lo cual hace sentido y por tanto estos valores perdidos **NO SON ERRORES**.
* Points_New_Version presenta casos con el texto "ND", lo cual estaría ocasionando que el tipo de variable no sea el correcto. En todo caso, sucede algo similar con lo visto en el punto anterior, estos "valores ausentes" no deberían considerarse errores al corresponder a fechas donde se aplicaban metodologías anteriores.

Mira más a fondo otras columnas. Por ejemplo, observa los valores posibles de Confederation usando los métodos `unique` y luego `value_counts`.

In [5]:
df_fifa["Confederation"].unique()

array(['UEFA', 'CONMEBOL', 'CONCACAF', 'CAF', 'AFC', nan], dtype=object)

In [6]:
df_fifa["Confederation"].value_counts(dropna = False)

Confederation
UEFA        14934
CAF         14876
AFC         12481
CONCACAF     9664
NaN          2982
CONMEBOL     2860
Name: count, dtype: int64

Está faltando una confederación que corresponde a la de Oceanía (OFC). Esto lo intuimos porque en la muestra de datos obtenida antes, las filas con valores perdidos en Confederation corresponden a países de esta región del mundo.

Verifica lo anterior utilizando un filtro en estos datos.

In [7]:
#Forma 1
df_fifa[df_fifa["Confederation"].isna()]["Country"].unique() #el método isna() es equivalente a isnull()

array(['Australia', 'New Zealand', 'Fiji', 'Tahiti', 'Solomon Islands',
       'Vanuatu', 'Papua New Guinea', 'Tonga', 'Samoa', 'Cook Islands',
       'American Samoa', 'New Caledonia'], dtype=object)

In [8]:
#Forma 2
df_fifa.loc[df_fifa["Confederation"].isna(),"Country"].unique()

array(['Australia', 'New Zealand', 'Fiji', 'Tahiti', 'Solomon Islands',
       'Vanuatu', 'Papua New Guinea', 'Tonga', 'Samoa', 'Cook Islands',
       'American Samoa', 'New Caledonia'], dtype=object)

Utiliza ahora el método `describe` para generar un resumen estadístico de la columna rank. 

In [9]:
df_fifa["rank"].describe()

count    57797.000000
mean       101.629514
std         58.618560
min          1.000000
25%         51.000000
50%        101.000000
75%        152.000000
max        209.000000
Name: rank, dtype: float64

Notemos que en algunas mediciones se han alcanzado hasta 209 paises puesto que este es su valor máximo. De hecho, vale mencionar que la FIFA actualmente reconoce a 211 países por lo que podríamos concluir que algunos países comparten una misma posición.

Mantengamos esa curiosidad por conocer más de los datos. Cuenta ahora cuántos paises fueron rankeados en cada una de las fechas. Ordena este conteo de mayor a menor usando el método `sort_values`.

In [10]:
df_fifa["Rank_Date"].value_counts().sort_values(ascending = False)

Rank_Date
2018-06-07    211
2017-02-09    211
2017-07-06    211
2018-05-17    211
2017-09-14    211
             ... 
1993-12-23    168
1993-10-22    167
1993-09-23    167
1994-02-15    167
1993-08-08    167
Name: count, Length: 286, dtype: int64

Comprueba finalmente si existen casos duplicados en el dataset. Considerando que existe una gran cantidad de registros en el dataset (aproximadamente 58K filas), por lo que no sería posible hacer una comprobación manual; por tanto utiliza el método `duplicated`.

In [11]:
df_fifa.duplicated().sum()

41

Mira en mayor detalle estos duplicados para saber de qué se tratan.

In [12]:
df_fifa[df_fifa.duplicated()]

,Country,Confederation,Rank_Date,Points_Old_Version,Points_New_Version,rank
14963,Finland,UEFA,2001-01-17,530.0,ND,60
23319,United Arab Emirates,AFC,2004-07-07,516.0,ND,71
34142,Samoa,NaN,2008-12-17,64.0,ND,176
41323,Sudan,CAF,2011-12-21,NaN,297.13,113
41539,Sudan,CAF,2012-01-18,NaN,264.81,120
41739,Sudan,CAF,2012-02-15,NaN,301.67,111
41947,Sudan,CAF,2012-03-07,NaN,299.75,110
42160,Sudan,CAF,2012-04-11,NaN,292.03,113
42369,Sudan,CAF,2012-05-09,NaN,292.03,113
42565,Sudan,CAF,2012-06-06,NaN,338.25,101


Gran parte de estos duplicados corresponden a calificaciones asociadas a la selección de Sudán.

En general, los valores duplicados pueden considerarse errores en el registro de información y nos interesa eliminar estos casos a fin de evitar distorsiones en los futuros resultados del análisis.

A partir de todo este entendimiento de los datos, establece un plan de acción para procesar esta información de modo que la tabla final resultante esté lista para cualquier análisis que desees hacer.

**ESCRIBE AQUÍ TU RESPUESTA**

* Los nombres de las columnas no están bien presentados, en concreto rank esta en minúsculas y el resto no. Se debería estandarizar a un mismo formato que siga el esquema *snake_case*.
* Las columna Confederation presenta valores ausentes que pueden corregirse inputando el valor "OFC" debido a que se identificó que estos son los países a los que corresponden dichos errores en la información.
* La columna Poins_New_Version debe cambiarse a un tipo float que responde más a su naturaleza de valores numéricos. Con este propósito en primera instancia conviene dar tratamiento a los casos "ND" detectados cambiándolos por valores ausentes.
* Las columnas Points_Old_Version y Point_New_Version igualmente presentan valores ausentes, sin embargo por el contexto del caso estos no serían errores sino que responden a los cambios metodológicos generados por la FIFA. En todo caso convendría crear una nueva columna que consolide esta información y que por tanto ya no tenga casos perdidos.  
* La columna Rank_Date debe cambiarse al tipo fecha (datetime). A partir de esto, se podría contar con campos adicionales de años y meses considerando que los datos mentienen generalmente esta periodicidad.
* Existen 41 filas duplicadas que deben eliminarse considerando que son errores en el dataset y afectarían los resultados de futuros análisis.

## Preparación de los datos

Utilizando nuestro plan de acción, procedamos a dejar listo este dataset para los análisis futuros que querramos hacer.

Entonces, en primer lugar ajusta los nombres de las columnas para que estén en formato *snake_case*. Recuerda que este formato exige que todos los nombres se encuentren en minúsculas y los espacios se sustituyan por "_".

In [13]:
nombres_sc = []
for col in df_fifa.columns:
    nombres_sc.append(col.lower().replace(" ","_"))

df_fifa.columns = nombres_sc
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             57797 non-null  object 
 1   confederation       54815 non-null  object 
 2   rank_date           57797 non-null  object 
 3   points_old_version  40382 non-null  float64
 4   points_new_version  57797 non-null  object 
 5   rank                57797 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 2.6+ MB


Otra forma más elegante y eficiente para hacer esto es mediante una sintaxis llamada *Python Comprehension* que dejaría todo el código anterior así:

```py
df_fifa.columns = [col.lower().replace(" ","_") for col in df_fifa.columns]
```

Como siguiente paso, inputa el valor "OFC" a los casos ausentes de la columna confederation. Utiliza para esto el método `fillna`.

In [14]:
df_fifa["confederation"] = df_fifa["confederation"].fillna("OFC")
df_fifa["confederation"].value_counts()

confederation
UEFA        14934
CAF         14876
AFC         12481
CONCACAF     9664
OFC          2982
CONMEBOL     2860
Name: count, dtype: int64

Ajusta ahora la columna points_new_version realizando lo siguiente:

* Cambia los valores perdidos "ND" por un valor perdido adecuado usando el valor `np.nan`.
* Cambia el tipo de variable a float utilizando el método `astype`.

In [15]:
# Cambiar valores perdidos ND por Nan
df_fifa.loc[df_fifa['points_new_version'] == 'ND','points_new_version'] = np.nan

In [16]:
# Cambiar tipo de variable
df_fifa['points_new_version'] = df_fifa['points_new_version'].astype(float)
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             57797 non-null  object 
 1   confederation       57797 non-null  object 
 2   rank_date           57797 non-null  object 
 3   points_old_version  40382 non-null  float64
 4   points_new_version  17415 non-null  float64
 5   rank                57797 non-null  int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 2.6+ MB


Consolida las columnas points_old_version y points_new_version en una nueva columna llamada *points*, de forma que ya no existan valores ausentes por los cambios metodológicos.

In [17]:
df_fifa.loc[df_fifa['points_old_version'].isna(), 'points'] = df_fifa['points_new_version']
df_fifa.loc[df_fifa['points_new_version'].isna(), 'points'] = df_fifa['points_old_version']
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             57797 non-null  object 
 1   confederation       57797 non-null  object 
 2   rank_date           57797 non-null  object 
 3   points_old_version  40382 non-null  float64
 4   points_new_version  17415 non-null  float64
 5   rank                57797 non-null  int64  
 6   points              57797 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 3.1+ MB


Para evitar confusiones posteriores, elimina las columnas points_old_version y points_new_version. Utiliza el método `drop`.

In [18]:
df_fifa = df_fifa.drop(columns = ["points_old_version","points_new_version"])
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        57797 non-null  object 
 1   confederation  57797 non-null  object 
 2   rank_date      57797 non-null  object 
 3   rank           57797 non-null  int64  
 4   points         57797 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 2.2+ MB


Transforma la columna rank_date a formato de fecha (datetime) aplicando la función `pd.to_datetime`.

In [19]:
df_fifa["rank_date"] = pd.to_datetime(df_fifa["rank_date"])
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   country        57797 non-null  object        
 1   confederation  57797 non-null  object        
 2   rank_date      57797 non-null  datetime64[ns]
 3   rank           57797 non-null  int64         
 4   points         57797 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 2.2+ MB


Ahora que esta columna ya es una fecha, es posible extraer sus componentes fundamentales. En este setido, crea una columna *year* y otra *month* que contenga la información correspondiente. 

In [20]:
# Columna year
df_fifa['year'] = df_fifa['rank_date'].dt.year

In [21]:
# Columna month
df_fifa['month'] = df_fifa['rank_date'].dt.month
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   country        57797 non-null  object        
 1   confederation  57797 non-null  object        
 2   rank_date      57797 non-null  datetime64[ns]
 3   rank           57797 non-null  int64         
 4   points         57797 non-null  float64       
 5   year           57797 non-null  int32         
 6   month          57797 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(2), int64(1), object(2)
memory usage: 2.6+ MB


Adicionalmente, y si bien no está en el plan de acción, podría ser útil especificar en una columna nueva la metodología utilizada para el cálculo de puntos. Crea por tanto la columna *method* la cual cumpla con los siguientes criterios secuenciales:

* Si el año y mes de la fila es menor a Enero 1999 entonces su valor será "1993 - 1998".
* Si el año y mes de la fila es menor a Agosto 2006 entonces su valor será "1999 - 2006"
* Si el año y mes de la fila es menor a Agosto 2011 entonces su valor será "2006 - 2011"
* Todos los demás casos deberían tener el valor "2011 - 2018"

In [22]:
df_fifa["method"] = ""

for i in range(df_fifa.shape[0]):
    periodomes = df_fifa.loc[i,"year"]*100 + df_fifa.loc[i,"month"]
    if periodomes < 199901:
        df_fifa.loc[i,"method"] = "1993-1998"
    elif periodomes < 200608:
        df_fifa.loc[i,"method"] = "1999-2006"
    elif periodomes < 201108:
        df_fifa.loc[i,"method"] = "2006-2011"
    else:
        df_fifa.loc[i,"method"] = "2011-2018"

df_fifa["method"].value_counts().sort_index()

method
1993-1998    10058
1999-2006    18113
2006-2011    12211
2011-2018    17415
Name: count, dtype: int64

Una forma más eficiente para realizar esto es mediante el método `apply`. Te sugiero que investigues un poco sobre esta función de **Pandas**.

Finalmente, da tratamiento a los duplicados eliminándolos con el método `drop_duplicates`.

In [23]:
df_fifa = df_fifa.drop_duplicates().reset_index(drop = True) #En general siempre es recomendable resetear los índices cuando se borran filas 
df_fifa.duplicated().sum()

0

Listo, hemos concluido con nuestro plan de acción. Dale una mirada nuevamente al dataframe limpio extrayendo una muestra de 15 elementos.

In [24]:
df_fifa.sample(15)

,country,confederation,rank_date,rank,points,year,month,method
46038,Scotland,UEFA,2013-11-28,33,716.57,2013,11,2011-2018
27116,Mongolia,AFC,2006-01-18,179,137.00,2006,1,1999-2006
52170,Mauritania,CAF,2016-04-07,103,333.95,2016,4,2011-2018
8968,Namibia,CAF,1998-07-15,81,36.00,1998,7,1993-1998
51050,Northern Ireland,UEFA,2015-11-05,29,797.29,2015,11,2011-2018
24025,Eritrea,CAF,2004-10-06,164,228.00,2004,10,1999-2006
34319,Mauritania,CAF,2009-01-14,149,112.00,2009,1,2006-2011
32099,Montserrat,CONCACAF,2008-02-13,202,0.00,2008,2,2006-2011
23696,Iraq,AFC,2004-09-01,40,601.00,2004,9,1999-2006
23811,Kyrgyz Republic,AFC,2004-09-01,155,263.00,2004,9,1999-2006


## Análisis de datos

Es hora que realices tu primer análisis de datos ahora que tienes una tabla con información limpia. Para un correcto análisis vale que tengas siempre en cuenta lo siguiente:

* Su propósito fundamental es **responder preguntas de negocio**, por lo que siempre debes buscar dar rsoluciones concretas en base a tus resultados.
* No se limita a realizar gráficos o tablas resumen, sino a generar **conclusiones** que aporten valor a partir de estas visualizaciones.
* Debe mantener un **hilo de ideas** conductor lo cual implica orden y consistencia entre los resultados.

Visto esto, a continuación verás algunas preguntas de negocio a responder: 

### ¿Cómo ha sido el comportamiento anual del puntaje de la CONMEBOL a partir de la última metodología?

En primer lugar identifica entre las columnas de la tabla cuál es la métrica relevante que debes obtener:

**ESCRIBE AQUÍ TU RESPUESTA**

A partir de la columna points, la métrica sería el puntaje promedio por año.

Ahora, especifica si debes realizar filtros en la tabla a partir de algunas condiciones:

**ESCRIBE AQUÍ TU RESPUESTA**

* Se debe filtrar la columna confederation manteniendo solamente las filas de "CONMEBOL".
* Se debe filtrar la culumna method manteniendo solamente las filas de "2011-2018"

Para terminar, indica si debes agregar los datos por alguna de las columnas: 

**ESCRIBE AQUÍ TU RESPUESTA**

Los datos deben agregarse por la columna year.

Construye el código necesario para responder la pregunta utilizando como criterios los aspectos contestados antes.

In [25]:
datos_filtrados = df_fifa.query("confederation == 'CONMEBOL' and method == '2011-2018'")
puntos_ec = datos_filtrados.groupby("year")["points"].mean()
puntos_ec

year
2011     818.367000
2012     848.170750
2013     891.147917
2014     920.097000
2015     912.779833
2016     996.969917
2017    1035.541833
2018     959.854500
Name: points, dtype: float64

Genera una conclusión relevante. Recuerda que una conclusión no se limita a una descripción de lo que obtuviste, sino a implicancias, hipótesis o recomendaciones a partir de estos resultados.

**ESCRIBE AQUÍ TU RESPUESTA**

* Se observa un crecimiento sostenido en el puntaje de las selecciones de CONMEBOL en este periodo lo cual está asociado al mejoramiento de selecciones no tan protagonistas como son Chile y Ecuador. En concreto, esto se confirma con las Copas América ganadas por Chile en los años 2015 y 2016, así como con los éxitos de clubes ecuatorianos a nivel internacional como IDV y LDU, los cuales son la base actual del seleccionado nacional.

* A pesar de esto, en el último año se evidencia una reducción significativa en el puntaje medio de CONMEBOL, explicado seguramente por los malos resultados de varias selecciones de esta confederación previo al mundial de Rusia de 2018:
    * Argentina perdió dos partidos importantes en su etapa preparatoria (2 - 4 versus Nigeria y 1 - 6 versus España).
    * Colombia perdió un partido preparatorio con Corea del Sur por (1 - 2) y empató otro contra Australia (0 - 0).
    * Chile y Ecuador quedaron eliminadas en la fase clasificatoria a pesar de estar bien posicionadas y en constante crecimiento los últimos años.
    * Bolivia recibió sanciones por parte de la FIFA producto de asuntos administrativos en su ente federativo nacional.
    

### ¿Cual ha sido la confederación peor evaluada en los últimos 3 años registrados?

Establece cuál es la métrica de análisis, los filtros y agrupamientos que necesitas y da una respuesta a la pregunta.

In [26]:
datos_filtrados = df_fifa.query("year >= 2016")
puntos_por_conf = datos_filtrados.pivot_table(
    columns = "year",
    index = "confederation",
    values = "points",
    aggfunc = "mean"
)
puntos_por_conf.round(1) #El método round permite una mejor visualización de los resultados

year,2016,2017,2018
confederation,,,
AFC,252.8,259.4,258.7
CAF,374.9,367.8,341.2
CONCACAF,290.7,278.1,253.6
CONMEBOL,997.0,1035.5,959.9
OFC,98.9,110.2,105.1
UEFA,644.9,642.8,647.6


**ESCRIBE AQUÍ UNA CONCLUSIÓN**

* OFC es la confederación con el peor posicionamiento entre las confederaciones lo cual se explica por el bajo nivel de fútbol competitivo y profesional que existe en los paises de Oceanía. Esto se acrecentó incluso más el año 2006 en el que Australia se afilió a la AFC para mantener un mayor nivel en sus enfrentamientos deportivos.